In [ ]:
import asyncio
import dotenv
from agents import (
    Agent,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    Runner,
    SQLiteSession,
    GuardrailFunctionOutput,
    RunContextWrapper,
    handoff,
    input_guardrail,
    output_guardrail,
)
from pydantic import BaseModel, Field

dotenv.load_dotenv()

model = "gpt-5.4-mini"

session = SQLiteSession(db_path="restaurant.db", session_id="Patrick Jane")


class CustomerInfo(BaseModel):
    customer_name: str = Field(..., description="고객 이름")


# input guardrail agent
class InputGuardrailOutput(BaseModel):
    is_not_valid: bool = Field(
        ...,
        description="레스토랑에 관한 질문인지 여부, 레스토랑에 관한 질문이 아니면 True, 레스토랑에 관한 질문이면 False",
    )
    reason: str = Field(..., description="이유")


input_guardrail_agent = Agent(
    name="input_guardrail_agent",
    instructions="사용자의 입력이 레스토랑(메뉴, 예약, 위치, 영업시간, 음식 등)에 관한 질문인지 판단해라.",
    output_type=InputGuardrailOutput,
    model=model,
)


@input_guardrail
async def off_topic_guardrail(
    wrapper: RunContextWrapper[CustomerInfo],
    agent: Agent[CustomerInfo],
    input: str,
) -> GuardrailFunctionOutput:
    result = await Runner.run(
        input_guardrail_agent,
        input=input,
        context=wrapper.context,
    )
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_not_valid,
    )


# output guardrail agent
class OutputGuardrailOutput(BaseModel):
    is_not_polite: bool = Field(
        ...,
        description="사용자의 입력이 예의바른지 여부, 예의바르지 않으면 True, 예의바르면 False",
    )
    reason: str = Field(..., description="이유")


output_guardrail_agent = Agent(
    name="output_guardrail_agent",
    instructions="""에이전트의 응답이 고객에게 불쾌감을 주는 단어가 있는지 판단해라. 불쾌감을 주는 단어가 있으면 True, 없으면 False""",
    output_type=OutputGuardrailOutput,
    model=model,
)


@output_guardrail
async def rude_words_guardrail(
    wrapper: RunContextWrapper[CustomerInfo],
    agent: Agent[CustomerInfo],
    input: str,
) -> GuardrailFunctionOutput:
    result = await Runner.run(
        output_guardrail_agent,
        input=input,
        context=wrapper.context,
    )
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_not_polite,
    )


# restaurant information agent
restaurant_information_agent = Agent(
    name="restaurant_information_agent",
    instructions="""너는 레스토랑의 정보를 제공하는 에이전트야.""",
    model=model,
)

# complaints agent
complaints_agent = Agent(
    name="complaints_agent",
    instructions="""You are a helpful assistant that can help customers with complaints about the restaurant.""",
    model=model,
)


# triage agent
class TriageHandOffInput(BaseModel):
    to_agent: str = Field(
        ...,
        description="이동할 에이전트 이름, restaurant_information_agent 또는 complaints_agent 중 하나",
    )
    input: str = Field(..., description="이동할 에이전트에 전달할 입력")
    reason: str = Field(..., description="해당 에이전트로 이동한 이유")


def on_handoff(
    wrapper: RunContextWrapper[CustomerInfo], input: TriageHandOffInput
) -> Agent[CustomerInfo]:
    print("=======================handoff input=========================")
    print(input)


def create_handoff(agent: Agent[CustomerInfo]):
    return handoff(
        agent=agent,
        on_handoff=on_handoff,
        input_type=TriageHandOffInput,
    )


triage_agent = Agent(
    name="triage_agent",
    instructions="""너는 고객의 질문을 분석하고, 고객의 질문에 따라 적절한 에이전트로 이동하는 에이전트야.""",
    handoffs=[
        create_handoff(restaurant_information_agent),
        create_handoff(complaints_agent),
    ],
    input_guardrails=[off_topic_guardrail],
    output_guardrails=[rude_words_guardrail],
    model=model,
)


async def main():
    try:
        result = await Runner.run(
            starting_agent=triage_agent,
            input="음식이 너무 맛이 없었어. 뭘 해줄수 있어?",
            context=CustomerInfo(customer_name="Patrick Jane"),
            session=session,
        )
        print(result.final_output)

    except InputGuardrailTripwireTriggered:
        print(
            "저는 레스토랑 관련 질문에 대해서만 도와드리고 있어요. 메뉴를 확인하거나, 예약하거나, 음식을 주문할 수 있어요."
        )

    except OutputGuardrailTripwireTriggered:
        print("잠시 후 다시 시도해주세요.")


if __name__ == "__main__":
    asyncio.run(main())